In [2]:
import numpy as np
import nibabel as nib
import os
import time
import torch
import torch.nn as nn
from torch.optim import AdamW
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import json

In [3]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cpu


In [4]:
"""
"tensorImageSize": "4D",
"modality": {
	 "0": "FLAIR",
	 "1": "T1w",
	 "2": "t1gd",
	 "3": "T2w"
 },
 "labels": {
	 "0": "background",
	 "1": "edema",
	 "2": "non-enhancing tumor",
	 "3": "enhancing tumour"
 },
"""
#mount data drive
from google.colab import drive

drive.mount('/content/drive') # mount drive
braintumor_datadir = "/content/drive/MyDrive/data/med/braintumor0" # load braintumor data directory
braintumor_dataimages = os.path.join(braintumor_datadir, "imagesTr") # load brain tumor imagees directory
braintumor_datalabels = os.path.join(braintumor_datadir, "labelsTr") # load brain tumor labels directory



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# Brats Brain Tumor dataset stats:
BRAINTUMOR_GLOBAL_STATS = {
    'FLAIR': {'p0.5': 23.958, 'p99.5': 1168.326},
    'T1w': {'p0.5': 45.15, 'p99.5': 2012.474},
    'T1GD': {'p0.5': 38.086, 'p99.5': 2305.63},
    'T2w': {'p0.5': 41.618, 'p99.5': 1669.87}
}

In [6]:
import numpy as np
import torch
from torch.utils.data import Dataset
import nibabel as nib
import os

def percentile_normalization_numpy(data_np, modality, stats_dict):
    """Normalize 3D numpy array using precomputed percentiles"""
    params = stats_dict[modality]
    p_low, p_high = params['p0.5'], params['p99.5']
    range_ = p_high - p_low

    # Initialize output
    normalized = np.zeros_like(data_np, dtype=np.float32)

    # Process foreground voxels
    mask = data_np > p_low
    foreground = data_np[mask]

    # Apply scaling
    scaled = (foreground - p_low) / range_
    scaled = np.clip(scaled, 0, 1)

    normalized[mask] = scaled
    return normalized

class BratsDataset(Dataset):
    def __init__(self, images_dir, labels_dir):
        super().__init__()
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.image_files = {f for f in os.listdir(images_dir) if f.endswith('.nii.gz')}
        self.label_files = {f for f in os.listdir(labels_dir) if f.endswith('.nii.gz')}
        self.common_files = sorted(self.image_files & self.label_files)
        self.modalities = ['FLAIR', 'T1w', 'T1GD', 'T2w']

        # Precomputed stats (replace with your actual values)
        self.global_stats = {
            'FLAIR': {'p0.5': 23.958, 'p99.5': 1168.326},
            'T1w': {'p0.5': 45.15, 'p99.5': 2012.474},
            'T1GD': {'p0.5': 38.086, 'p99.5': 2305.63},
            'T2w': {'p0.5': 41.618, 'p99.5': 1669.87}
        }

    def __len__(self):
        return len(self.common_files)

    def __getitem__(self, index):
        file_index = self.common_files[index]

        # Load data (shape: H x W x D x C)
        image_np = nib.load(os.path.join(self.images_dir, file_index)).get_fdata()
        label_np = nib.load(os.path.join(self.labels_dir, file_index)).get_fdata()

        # Normalize each channel in NumPy
        normalized_channels = []
        for c, modality in enumerate(self.modalities):
            channel_np = image_np[..., c]  # Extract channel (H,W,D)
            norm_channel = percentile_normalization_numpy(channel_np, modality, self.global_stats)
            normalized_channels.append(norm_channel)

        # Stack normalized channels and convert to tensor
        image_norm = np.stack(normalized_channels, axis=-1)  # H x W x D x C
        image_tensor = torch.from_numpy(image_norm).permute(3, 2, 0, 1).float()  # C x D x H x W

        # Process label
        label_tensor = torch.from_numpy(label_np).permute(2, 0, 1).long()  # D x H x W

        return image_tensor, label_tensor

In [7]:
bt_dataset = BratsDataset(braintumor_dataimages, braintumor_datalabels)
bt_dataloader = DataLoader(bt_dataset, batch_size=1, pin_memory=False)

In [8]:

def conv_norm_act(in_channels, out_channels, kernel=3, padding=1, stride=1):

    return nn.Sequential(
        nn.Conv3d(in_channels, out_channels, kernel_size=kernel, padding=padding, stride=stride),
        nn.GroupNorm(out_channels//4, out_channels),
        nn.SiLU()

    )

def conv_norm_act_upsample(in_channels, out_channels, target_size, kernel=3, padding=1, stride=1):

    return nn.Sequential(
        nn.Upsample(size=target_size, mode='trilinear', align_corners=True),
        nn.Conv3d(in_channels, out_channels, kernel_size=kernel, padding=padding, stride=stride),
        nn.GroupNorm(out_channels//4, out_channels),
        nn.SiLU()
    )


class UNet3D(nn.Module):
    def __init__(self, in_channels=4, num_classes=4, base_channels=8):
        super().__init__()
        chs = [base_channels * (2**i) for i in range(5)]  # [8, 16, 32, 64, 128]

        # Encoder (Downsampling path)
        self.encoder = nn.ModuleList([
            conv_norm_act(in_channels, chs[0]),              # (B, 4, D, H, W) → (B, 8, D, H, W)
            conv_norm_act(chs[0], chs[1], stride=2),          # (B, 8, ...) → (B, 16, D/2, H/2, W/2)
            conv_norm_act(chs[1], chs[2], stride=2),          # ... → (B, 32, D/4, H/4, W/4)
            conv_norm_act(chs[2], chs[3], stride=2),          # ... → (B, 64, D/8, H/8, W/8)
            conv_norm_act(chs[3], chs[4], stride=2)           # ... → (B, 128, D/16, H/16, W/16)
        ])

        # Bottleneck (Bottom layer)
        self.bottleneck = conv_norm_act(chs[4], chs[4] * 2)   # (B, 128, ...) → (B, 256, ...)

        # Decoder (Upsampling path with proper skip connections)
        self.decoder = nn.ModuleList([
            nn.Sequential(
                conv_norm_act_upsample(256, 128, target_size=(10, 15, 15)),  # (B, 256, ...) → (B, 128, ...)
                conv_norm_act(128 + 128, 128)                           # After cat: (B, 256, ...) → (B, 128, ...)
            ),
            nn.Sequential(
                conv_norm_act_upsample(128, 64, target_size=(20, 30, 30)),  # (B, 128, ...) → (B, 64, ...)
                conv_norm_act(64 + 64, 64)                              # After cat: (B, 128, ...) → (B, 64, ...)
            ),
            nn.Sequential(
                conv_norm_act_upsample(64, 32, target_size=(39, 60, 60)),   # (B, 64, ...) → (B, 32, ...)
                conv_norm_act(32 + 32, 32)                             # After cat: (B, 64, ...) → (B, 32, ...)
            ),
            nn.Sequential(
                conv_norm_act_upsample(32, 16, target_size=(78, 120, 120)), # (B, 32, ...) → (B, 16, ...)
                conv_norm_act(16 + 16, 16)                             # After cat: (B, 32, ...) → (B, 16, ...)
            ),
            nn.Sequential(
                conv_norm_act_upsample(16, 8, target_size=(155, 240, 240)), # (B, 16, ...) → (B, 8, ...)
                conv_norm_act(8 + 8, 8)                                # After cat: (B, 16, ...) → (B, 8, ...)
            )
        ])

        # Final output with softmax for Dice
        self.final_conv = nn.Conv3d(8, num_classes, kernel_size=1, padding=0)

    def forward(self, x):
        skips = []

        # Encoder
        for layer in self.encoder:
            x = layer(x)
            skips.append(x)

        # Bottleneck
        x = self.bottleneck(x)

        # Decoder (reverse skips)
        for i, (layer, skip) in enumerate(zip(self.decoder, reversed(skips))):
            x = layer[0](x)          # Upsample + conv
            x = torch.cat([x, skip], dim=1)  # Concatenate with skip
            x = layer[1](x)          # Merge features

        return self.final_conv(x)


In [9]:
import torch
import torch.nn as nn

# --- Utility conv blocks ---
def conv_norm_act(in_channels, out_channels, kernel=3, padding=1, stride=1):
    return nn.Sequential(
        nn.Conv3d(in_channels, out_channels, kernel_size=kernel, padding=padding, stride=stride),
        nn.GroupNorm(out_channels // 4, out_channels),
        nn.SiLU()
    )

def conv_norm_act_upsample(in_channels, out_channels, target_size, kernel=3, padding=1, stride=1):
    return nn.Sequential(
        nn.Upsample(size=target_size, mode='trilinear', align_corners=True),
        nn.Conv3d(in_channels, out_channels, kernel_size=kernel, padding=padding, stride=stride),
        nn.GroupNorm(out_channels // 4, out_channels),
        nn.SiLU()
    )

class UNet3DResidual(nn.Module):
    def __init__(self):
        super().__init__()

        # --- Encoder ---
        self.encoder_1  = conv_norm_act(4, 4)                 # (4, 155, 240, 240) -> (4, 155, 240, 240)
        self.encoder_2  = conv_norm_act(4, 8, stride=2)        # (4, 155, 240, 240) -> (8, 78, 120, 120)
        self.encoder_3  = conv_norm_act(8, 8)                  # (8, 78, 120, 120) -> (8, 78, 120, 120)
        self.encoder_4  = conv_norm_act(8, 16, stride=2)       # (8, 78, 120, 120) -> (16, 39, 60, 60)
        self.encoder_5  = conv_norm_act(16, 16)                # (16, 39, 60, 60) -> (16, 39, 60, 60)
        self.encoder_6  = conv_norm_act(16, 32, stride=2)      # (16, 39, 60, 60) -> (32, 20, 30, 30)
        self.encoder_7  = conv_norm_act(32, 32)                # (32, 20, 30, 30) -> (32, 20, 30, 30)
        self.encoder_8  = conv_norm_act(32, 64, stride=2)      # (32, 20, 30, 30) -> (64, 10, 15, 15)
        self.encoder_9  = conv_norm_act(64, 64)                # (64, 10, 15, 15) -> (64, 10, 15, 15)
        self.encoder_10 = conv_norm_act(64, 128)               # (64, 10, 15, 15) -> (128, 10, 15, 15)
        self.encoder_11 = conv_norm_act(128, 128)              # (128, 10, 15, 15) -> (128, 10, 15, 15)
        self.bottleneck = conv_norm_act(128, 256)              # (128, 10, 15, 15) -> (256, 10, 15, 15)

        # --- Decoder ---
        # Stage 1: from bottleneck, skip(128, 10, 15, 15)
        self.dec1_conv1 = conv_norm_act(256, 128)
        self.dec1_conv2 = conv_norm_act(128, 128)              # residual
        self.dec1_up    = conv_norm_act_upsample(128 + 128, 128, (20, 30, 30))

        # Stage 2: skip(32, 20, 30, 30)
        self.dec2_conv1 = conv_norm_act(128, 64)
        self.dec2_conv2 = conv_norm_act(64, 64)
        self.dec2_up    = conv_norm_act_upsample(64 + 32, 64, (39, 60, 60))

        # Stage 3: skip(16, 39, 60, 60)
        self.dec3_conv1 = conv_norm_act(64, 32)
        self.dec3_conv2 = conv_norm_act(32, 32)
        self.dec3_up    = conv_norm_act_upsample(32 + 16, 32, (78, 120, 120))

        # Stage 4: skip(8, 78, 120, 120)
        self.dec4_conv1 = conv_norm_act(32, 16)
        self.dec4_conv2 = conv_norm_act(16, 16)
        self.dec4_up    = conv_norm_act_upsample(16 + 8, 16, (155, 240, 240))

        # Final output
        self.final_conv = nn.Conv3d(16 + 4, 4, kernel_size=1)

    def forward(self, x):
        skips = []

        # --- Encoder ---
        x1 = self.encoder_1(x)                                # (4, 155, 240, 240)
        x1 = x1 + x                                           # residual
        skips.append(x1)                                      # skip0: (4, 155, 240, 240)

        x2 = self.encoder_2(x1)                               # (8, 78, 120, 120)
        x3 = self.encoder_3(x2)                               # (8, 78, 120, 120)
        x3 = x3 + x2
        skips.append(x3)                                      # skip1: (8, 78, 120, 120)

        x4 = self.encoder_4(x3)                               # (16, 39, 60, 60)
        x5 = self.encoder_5(x4)                               # (16, 39, 60, 60)
        x5 = x5 + x4
        skips.append(x5)                                      # skip2: (16, 39, 60, 60)

        x6 = self.encoder_6(x5)                               # (32, 20, 30, 30)
        x7 = self.encoder_7(x6)                               # (32, 20, 30, 30)
        x7 = x7 + x6
        skips.append(x7)                                      # skip3: (32, 20, 30, 30)

        x8 = self.encoder_8(x7)                               # (64, 10, 15, 15)
        x9 = self.encoder_9(x8)                               # (64, 10, 15, 15)
        x9 = x9 + x8

        x10 = self.encoder_10(x9)                             # (128, 10, 15, 15)
        x11 = self.encoder_11(x10)                            # (128, 10, 15, 15)
        x11 = x11 + x10
        skips.append(x11)                                     # skip4: (128, 10, 15, 15)

        # --- Bottleneck ---
        xb = self.bottleneck(x11)                             # (256, 10, 15, 15)

        # --- Decoder ---
        # Stage 1
        xd1 = self.dec1_conv1(xb)                             # (128, 10, 15, 15)
        xd1 = self.dec1_conv2(xd1) + xd1                      # residual
        xd1 = torch.cat([xd1, skips[4]], dim=1)               # (128+128=256, 10, 15, 15)
        xd1 = self.dec1_up(xd1)                               # (128, 20, 30, 30)

        # Stage 2
        xd2 = self.dec2_conv1(xd1)                            # (64, 20, 30, 30)
        xd2 = self.dec2_conv2(xd2) + xd2
        xd2 = torch.cat([xd2, skips[3]], dim=1)               # (64+32=96, 20, 30, 30)
        xd2 = self.dec2_up(xd2)                               # (64, 39, 60, 60)

        # Stage 3
        xd3 = self.dec3_conv1(xd2)                            # (32, 39, 60, 60)
        xd3 = self.dec3_conv2(xd3) + xd3
        xd3 = torch.cat([xd3, skips[2]], dim=1)               # (32+16=48, 39, 60, 60)
        xd3 = self.dec3_up(xd3)                               # (32, 78, 120, 120)

        # Stage 4
        xd4 = self.dec4_conv1(xd3)                            # (16, 78, 120, 120)
        xd4 = self.dec4_conv2(xd4) + xd4
        xd4 = torch.cat([xd4, skips[1]], dim=1)               # (16+8=24, 78, 120, 120)
        xd4 = self.dec4_up(xd4)                               # (16, 155, 240, 240)

        # Final stage
        out = torch.cat([xd4, skips[0]], dim=1)               # (16+4=20, 155, 240, 240)
        out = self.final_conv(out)                            # (4, 155, 240, 240)

        return out


In [10]:

class WeightedDiceLoss(nn.Module):
    def __init__(self, num_classes=4,
                 false_positive_weights=None, false_negative_weights=None,
                 class_importance_weights=None, epsilon=1e-6):
        """
        Args:
            num_classes (int): number of segmentation classes
            false_positive_weights (Tensor): shape [C], λ^FP_c per class
            false_negative_weights (Tensor): shape [C], λ^FN_c per class
            class_importance_weights (Tensor, optional): shape [C], γ_c or w_c
            epsilon (float): smoothing to avoid division by zero
        """
        super().__init__()
        self.num_classes = num_classes
        # Register weights as buffers so they are moved to the device with the module
        self.register_buffer('fp_weights', false_positive_weights)
        self.register_buffer('fn_weights', false_negative_weights)
        self.register_buffer('class_weights', class_importance_weights)
        self.epsilon = epsilon

    def forward(self, logits, targets):
        """
        Args:
            logits: Tensor of shape [B, C, D, H, W]
            targets: Tensor of shape [B, D, H, W] with values in 0,...,C-1
        Returns:
            Scalar loss value
        """
        B, C, *spatial = logits.shape
        probs = F.softmax(logits, dim=1)  # Step 1: Softmax
        targets_onehot = F.one_hot(targets, num_classes=C).permute(0, 4, 1, 2, 3).float()  # Step 2: One-hot

        # Step 3: Compute TP, FP, FN
        TP = (probs * targets_onehot).sum(dim=[0, 2, 3, 4])
        FP = (probs * (1 - targets_onehot)).sum(dim=[0, 2, 3, 4])
        FN = ((1 - probs) * targets_onehot).sum(dim=[0, 2, 3, 4])

        # Step 4: Compute class frequency weights if not provided
        if self.class_weights is None:
            class_counts = targets_onehot.sum(dim=[0, 2, 3, 4])
            # Ensure class_weights are on the same device as other tensors
            class_weights = (1.0 / (class_counts + self.epsilon)).to(logits.device)
            class_weights = class_weights / class_weights.sum()
        else:
            class_weights = self.class_weights


        # Step 5-6: Compute Tversky loss per class
        numerator = TP
        denominator = TP + self.fp_weights * FP + self.fn_weights * FN + self.epsilon
        class_losses = 1.0 - numerator / denominator  # [C]

        # Step 7: Weighted sum of losses
        total_loss = (class_weights * class_losses).sum()

        stats = {
        'TP': TP.detach().cpu(),
        'FP': FP.detach().cpu(),
        'FN': FN.detach().cpu()
    }

        return total_loss, stats

In [11]:
def compute_metrics_from_stats(stats):
    """
    Args:
        stats: dict with keys 'TP', 'FP', 'FN' as tensors of shape [C]
    Returns:
        dict: class_idx -> precision, recall, etc.
    """
    TP = stats['TP']
    FP = stats['FP']
    FN = stats['FN']

    metrics = {}
    for c in range(len(TP)):
        tp = TP[c].item()
        fp = FP[c].item()
        fn = FN[c].item()

        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)

        metrics[c] = {
            'TP': tp,
            'FP': fp,
            'FN': fn,
            'precision': precision,
            'recall': recall
        }

    return metrics


In [12]:
class TumorFocusedWeightedDiceLoss(WeightedDiceLoss):
    def __init__(self, gamma=2.0, tumor_classes=[1,2,3], **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma  # Focusing parameter
        self.tumor_classes = tumor_classes

    def forward(self, logits, targets):
        total_loss, stats = super().forward(logits, targets)

        # Compute class_losses within this class
        TP = stats['TP'].to(logits.device)
        FP = stats['FP'].to(logits.device)
        FN = stats['FN'].to(logits.device)

        numerator = TP
        denominator = TP + self.fp_weights * FP + self.fn_weights * FN + self.epsilon
        class_losses = 1.0 - numerator / denominator  # [C]

        # Compute tumor-class-only loss
        tumor_mask = torch.zeros(self.num_classes, dtype=torch.bool).to(logits.device)
        for c in self.tumor_classes:
            tumor_mask[c] = True

        tumor_loss = 0
        for c in self.tumor_classes:
            dice_score = 1 - class_losses[c]
            tumor_loss += (1 - dice_score) ** self.gamma

        # Combine with background loss (original weight)
        # Ensure class_weights are on the same device
        """
        if self.class_weights is None:

        else:
             class_weights = self.class_weights.to(logits.device)
        """

        class_weights = (1.0 / (stats['TP'] + stats['FP'] + stats['FN'] + self.epsilon)).to(logits.device)
        class_weights = class_weights / class_weights.sum()

        background_weight = class_weights[0]
        combined_loss = background_weight * (1 - class_losses[0]) + tumor_loss

        # Update stats dictionary with class_losses
        stats['class_losses'] = class_losses.detach().cpu()


        return combined_loss, stats

In [13]:
model_w_dicev1 = UNet3DResidual()
optimizer_w_dicev1 = AdamW(model_w_dicev1.parameters(), lr=1e-4, weight_decay=1e-3)
num_classes = 4
fp_weights = torch.tensor([1.0] * num_classes)  # λ_FP
fn_weights = torch.tensor([1.0] * num_classes)  # λ_FN
class_weights = None  # or provide custom weights as tensor of shape [C]

# Pass the fp_weights and fn_weights tensors to the loss function
loss_dice_v1 = TumorFocusedWeightedDiceLoss(
    num_classes=num_classes,
    false_positive_weights=fp_weights,
    false_negative_weights=fn_weights,
    class_importance_weights=class_weights # This is handled internally if None, but explicitly passing is fine
)

In [14]:
def trainV1(model, train_loader, loss_fn, optimizer, num_iterations, save_dir,
          device='cuda', early_stop_patience=10, accumulation_steps=4): # Added accumulation_steps

    os.makedirs(save_dir, exist_ok=True)
    #model.to(device)
    #loss_fn.to(device)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer=optimizer,
        mode='min',
        factor=0.5,
        patience=3,
        verbose=True,
        min_lr=1e-6
    )

    logs = []
    optimizer.zero_grad() # Initialize gradients

    for step, (inputs, targets) in enumerate(train_loader):
        if step >= num_iterations:
            break

        start_time = time.time()

        model.train()
        #inputs, targets = inputs.to(device), targets.to(device)

        logits = model(inputs)  # [B, C, D, H, W]
        loss, stats = loss_fn(logits, targets)

        # Normalize loss by accumulation steps
        loss = loss / accumulation_steps

        loss.backward()

        # Perform optimizer step and clear gradients only after accumulation_steps
        if (step + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()
            scheduler.step(loss) # Step scheduler after optimizer update

        step_time = time.time() - start_time
        current_lr = optimizer.param_groups[0]['lr']
        class_metrics = compute_metrics_from_stats(stats)

        log_entry = {
            'step': step,
            'step_time_sec': round(step_time, 3),
            'loss': loss.item() * accumulation_steps, # Log the un-normalized loss
            'lr': current_lr,
            'class_stats': class_metrics
        }
        logs.append(log_entry)

        # Console Output
        print(f"[{step:04d}] {step_time:.2f}s | Loss: {log_entry['loss']:.4f} | LR: {current_lr:.2e}")
        for cls_id, cm in class_metrics.items():
            print(f"  Class {cls_id}: TP={cm['TP']}  FP={cm['FP']}  FN={cm['FN']}  "
                  f"Precision={cm['precision']:.3f}  Recall={cm['recall']:.3f}")

        # Save model + metrics log (optional, might want to save less frequently)
        # torch.save(model.state_dict(), os.path.join(save_dir, f'model_step{step}.pth'))
        with open(os.path.join(save_dir, 'metrics.json'), 'w') as f:
            json.dump(logs, f, indent=2)

    # Perform a final optimizer step if there are remaining gradients
    if (step + 1) % accumulation_steps != 0:
        optimizer.step()
        optimizer.zero_grad()

    return logs

In [15]:
trainV1(model_w_dicev1, bt_dataloader, loss_dice_v1, optimizer_w_dicev1, 50, "/dice_v1")

/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn